# 06. YOLO26n Pose → LLM → ROS / 105분

**선행:** 01–04 손 추종·판단·TTL, 05 detection·pose와 가장 큰 사람 선택

**이 노트북의 결과:** 선택한 사람의 박스·17개 관절 전체를 이미지 없이 LLM에 전달합니다. “코를 추종해”에 대한 응답을 검증하는 `validate_pose_decision()`을 구현하고 ROS 거북이가 같은 프레임의 코 위치를 따라가도록 연결합니다.

[단원 README](README.md) · [용어집](GLOSSARY.md)


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

LLM은 사진 대신 가장 크게 검출된 사람의 전체 관절 JSON과 자연어 지시를 받습니다. 코를 추종하라는 지시에서 관절 이름을 선택하고, 코드가 그 이름과 좌표가 실제 입력에 있는지 확인합니다. 목표 좌표를 거북이 속도로 바꾸는 역할은 공통 제어기에 남습니다.

### 전체 과정에서 내 코드가 하는 일

```text
YOLO pose
  ↓
최대 면적 한 사람·17개 관절
  ↓
텍스트 LLM
  ↓
응답 검증 TODO
  ↓
검출 원본 좌표
  ↓
ROS 제어
```

frame_id·관절 이름·confidence·좌표 오차·정상 STOP을 검사합니다. 허용 오차 안의 응답도 최종 좌표는 원본으로 복원합니다. 코→손목 실험은 instruction과 target_keypoint를 함께 맞춥니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 코만 고정 추종한다면 LLM이 반드시 필요한가요?

**설명:** 아닙니다. 코 좌표 직접 선택은 코드로도 가능합니다. 여기서는 자연어 지시 변경과 모델 출력 검증의 역할을 관찰합니다. 코 고정 규칙의 성능 측정 경로는 제공하지 않으므로 실행하지 않은 기준선의 지연·오판 수치를 만들어 비교하지 않습니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**4일차 환경:** `nepes-course:physical-ai-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 카메라·GUI는 로컬 화면에서 함께 관찰하고 입력 프로세스는 하나만 실행합니다.

**VLM 실습 준비:** VLM 질의에는 공용 Ollama의 `http://127.0.0.1:11434` API와 `gemma4:e2b-it-qat` 모델이 필요합니다. 해당 실습의 연결 확인에서 실패하면 로컬 Setup 담당자에게 서버·모델 준비를 요청합니다. 순수 Python fixture와 YOLO 실습에는 Ollama가 필요하지 않습니다.

**작업 순서:** 아래 첫 설정 셀의 `ROOT`가 `/workspace`인지 확인 → 현재 노트북이 지정한 `exercises/*.py`를 Jupyter 편집기에서 저장 → `student = load_file(...)` 검증 셀을 다시 실행합니다. 새 커널에는 이전 노트북의 변수나 import가 없으므로 이 노트북 자체의 설정·fixture 셀을 사용합니다. 실행 중 보이는 출력은 학습 확인용이고 배포본에는 저장하지 않습니다. TODO 실패와 카메라·모델 환경 실패를 구분하세요.

## 슬라이드와 함께 진행하기

시작 안내는 [4일차 슬라이드 63쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s014) · [배포 PDF 63쪽](../슬라이드/4일차.pdf#page=63)입니다. 노트북의 설명·TODO·검증·실제 실행을 진행한 뒤, [4일차 슬라이드 64쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_pose_llm_01) · [배포 PDF 64쪽](../슬라이드/4일차.pdf#page=64)로 돌아가 64–71쪽에서 결과를 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.


## 정답 활용 안내

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 이번 실습이 전체 흐름에서 맡는 역할

[01 규칙 이동](01_ros_gesture_action.ipynb) → [02 VLM 프롬프트 검증](02_vlm_decision.ipynb) → [03 VLM 손 추종](03_ros_vlm_action.ipynb) → [04 판단 유지시간](04_ros_vlm_latency.ipynb) → [05 YOLO detection·pose](05_yolo_detection_pose.ipynb) → [06 pose 좌표·LLM·ROS](06_ros_pose_llm.ipynb). [07 segmentation](07_yolo_segmentation.ipynb)은 선택 자습입니다.

**구현 실습과 실제 실행의 관계:** 지정한 `exercises/` 파일에 학생이 알고리즘을 구현하고 노트북 검증 셀로 평가합니다. `solutions/`는 별도 참고 정답이며, 참고 정답 검증만 실행해도 학생 TODO가 완성되는 것은 아닙니다. 카메라·ROS 데모는 제공된 실행용 구현을 사용합니다. 학생 파일 저장이 실행 중인 ROS 노드를 자동으로 바꾸지는 않습니다. 실시간 동작과 자신의 함수 검증을 각각 완료하세요.


## 시간 지도

전체 좌표·프롬프트 15분 → validator TODO·검증 25분 → 정답·실패 해석 15분 → 실제 ROS 30분 → 지시 관절 변경·결과 비교 15분 → 저장 5분


## 필요한 앞 내용 — 서로 다른 세 가지 좌표

1. YOLO는 **원본 영상 기준** bbox·17개 관절 좌표를 찾습니다. 정규화 x는 오른쪽, y는 아래로 증가합니다.
2. 가장 큰 사람을 고른 뒤 **박스와 모든 관절 이름·좌표·신뢰도** 를 LLM에 JSON으로 보냅니다. `images`는 보내지 않습니다. LLM의 역할은 자연어 요청에 대응하는 관절 선택과 FOLLOW/STOP 응답입니다.
3. 검증 코드가 같은 프레임의 해당 관절을 다시 조회합니다. LLM이 반환한 좌표가 검출 결과와 맞는지 검사하고, 실제 ROS 목표는 원본 검출 좌표로 `(1+9*x,1+9*(1-y))`를 계산합니다.

LLM은 없는 관절을 상상하거나 속도를 정하지 않습니다. `frame_id`·프레임 유효시간·TTL·현재 거북이 pose 검사는 앞 실습처럼 계속 적용합니다. 다음 판단이 늦으면 승인이 만료될 수 있고, 목표에 먼저 도착하면 TTL이 남아도 멈춥니다.

아래 fixture는 모델·카메라 없이 독립 실행됩니다. 기본 코 좌표 `[0.5,0.3]`의 거북이 목표는 `(5.5,7.3)`입니다.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib.util, json, shutil, sys

def find_course_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "09.Pseudo-VLA").is_dir():
            return candidate
    raise RuntimeError("09.Pseudo-VLA를 포함한 과정 루트를 찾지 못했습니다.")

ROOT = find_course_root()
LESSON = ROOT / "09.Pseudo-VLA"
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
OPTIONAL = {name: bool(importlib.util.find_spec(name)) for name in ('cv2', 'ultralytics', 'rclpy', 'requests')}
print({"course_root": str(ROOT), "optional_live_dependencies": OPTIONAL})

def load_file(module_name, relative_path):
    path = LESSON / relative_path
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


## 전체 좌표를 보내고 목표 선택을 받기

<!-- procedure:lesson-35-def0d1:start -->
![사람 전체 좌표 → LLM → ROS](../교재그림/절차/lesson-35.svg)

이번 LLM 요청에는 이미지가 없습니다. 좌표를 새로 만들지 않고 원본 관측의 선택 결과를 검증합니다.
<!-- procedure:lesson-35-def0d1:end -->


```text
카메라 → YOLO26n-pose → 가장 큰 사람의 전체 관절 JSON
                                ↓ 텍스트 요청 + “코를 추종해”
                               LLM
                                ↓ 엄격한 JSON
                같은 frame·관절·좌표·confidence 검사
                                ↓ 원본의 코 좌표
                    ROS 목표 변환·속도 제어·overlay
```

입력 관측은 `frame_id`, `captured_ms`, `person`입니다. `person`에는 bbox·person confidence와 17개 관절이 있고, 미검출 관절도 이름을 유지하며 `xy=None`으로 표시합니다. 가장 큰 사람이 없으면 `person=None`입니다. LLM 응답의 필드는 정확히 `command,keypoint,center,frame_id,reason` 다섯 개입니다.

| 응답 | command | keypoint | center |
|---|---|---|---|
| 코가 유효한 추종 | FOLLOW | nose | 같은 frame 코의 정규화 `[x,y]` |
| 코 불확실·사람 없음 | STOP | null | null |

person confidence는 “선택할 사람”, 관절 confidence는 “추종할 점”의 판단에 씁니다. 최대 면적 사람의 코가 불확실해도 작은 사람의 코로 자동 교체하지 않습니다. 파서가 거부한 응답도 최종 제어에서는 STOP입니다. 모든 STOP이 잘된 장면 판단인 것은 아니므로 불필요한 STOP은 별도로 기록합니다.

**전송 숫자 표현:** 실제 LLM 요청에서는 box·좌표·confidence를 소수점 넷째 자리로 반올림해 텍스트 길이를 줄입니다. 17개 관절과 누락 상태는 모두 유지하고 frame_id·captured_ms는 그대로 보냅니다. 원본 관측·로그는 전체 정밀도를 유지하며, 검증도 원본 confidence·좌표를 기준으로 수행합니다. 숫자의 전송 표현을 줄이는 것이 프레임 나이·TTL 제한이나 추종 근거를 완화하는 것은 아닙니다. reason은 짧게 요청하되 검증 계약은 기존 1–160자입니다.


In [ ]:
# 모델·카메라·ROS가 필요 없는 독립 fixture입니다.
from copy import deepcopy
COCO_KEYPOINTS = ["nose", "left_eye", "right_eye", "left_ear", "right_ear", "left_shoulder", "right_shoulder", "left_elbow", "right_elbow", "left_wrist", "right_wrist", "left_hip", "right_hip", "left_knee", "right_knee", "left_ankle", "right_ankle"]
def fixture_person(bbox, confidence, nose=(0.5, 0.3)):
    joints = {name: {"xy": [0.5, 0.5], "confidence": 0.9} for name in COCO_KEYPOINTS}
    joints["nose"]["xy"] = list(nose)
    return {"bbox": list(bbox), "confidence": confidence, "keypoints": joints}

PEOPLE = [
    fixture_person([0.2, 0.2, 0.4, 0.7], 0.95),   # 면적 0.10: 점수는 가장 높음
    fixture_person([0.1, 0.1, 0.7, 0.9], 0.80),   # 면적 0.48: 선택할 사람
    fixture_person([0.0, 0.0, 1.0, 1.0], 0.49),   # 면적 1.00: 점수 기준 미달
]
print("예측: index 1, bbox 면적 0.48; 최고 confidence와 최대 면적은 다른 기준입니다.")

POSE_OBSERVATION = {"frame_id":"pose-f1", "captured_ms":1000, "person":deepcopy(PEOPLE[1])}
POSE_DECISION = {"command":"FOLLOW", "keypoint":"nose", "center":[0.5,0.3], "frame_id":"pose-f1", "reason":"requested nose is visible"}
POSE_RAW = json.dumps(POSE_DECISION)
POSE_STOP = {"command":"STOP", "keypoint":None, "center":None, "frame_id":"pose-f1", "reason":"no valid nose"}
print(json.dumps(POSE_OBSERVATION, ensure_ascii=False, indent=2))
print("예상 응답:", POSE_RAW)


## TODO — LLM 응답을 관측에 대조해 검증

**작성 파일:** [exercises/pose_llm_contract.py](exercises/pose_llm_contract.py)의 `validate_pose_decision(raw, observation, *, expected_keypoint="nose", min_keypoint_conf=0.5, tolerance=0.002)`입니다. 아래 계약을 직접 구현합니다. 파싱 helper를 같은 파일에 작성할 수 있습니다.

**입력:** `raw`는 LLM 응답 문자열입니다. `observation`은 해당 요청에 보낸 관측 dict입니다. 기본 대상은 `nose`, 관절 점수 경계는 0.5, 좌표 오차 허용은 축마다 0.002입니다. `expected_keypoint`는 실습 설정값이고, 모델이 마음대로 바꾸는 필드가 아닙니다.

**출력:** 모든 검사를 통과하면 응답 dict, 아니면 `None`입니다. FOLLOW의 `center`는 반올림된 응답 값 대신 **관측 원본 좌표로 바꿔 반환** 합니다. 다른 필드는 파싱한 값을 유지합니다. 오류를 중앙 좌표나 가짜 FOLLOW로 복구하지 않습니다. 입력 `observation`을 수정하지 않습니다. 실제 모델 호출·사람 선택·속도 계산·ROS 발행은 이 TODO의 범위 밖입니다.

검사할 순서는 다음과 같습니다.

1. JSON object를 읽되 중복 key·코드 펜스·추가 문장·list·필드 누락·추가 필드를 거부합니다. 정확한 key는 다섯 개입니다.
2. command는 FOLLOW/STOP, frame_id와 reason은 비어 있지 않은 문자열입니다. reason은 공백 제거 후 1–160자이며, 응답 frame_id는 관측 frame_id와 같아야 합니다.
3. STOP이면 `keypoint=None`와 `center=None`이어야 합니다. 같은 frame의 정상 STOP은 person이 없어도 허용합니다.
4. FOLLOW이면 `keypoint==expected_keypoint`여야 합니다. 선택한 사람의 해당 관절이 있고, 그 관절 confidence가 기준 이상이어야 합니다. 값은 bool이 아닌 유한수, 좌표는 길이 2의 0–1 숫자여야 합니다.
5. 응답 center도 0–1 유한수 두 개여야 합니다. 각 축이 원본 관절 좌표와 `tolerance` 이하만큼 차이 나는지 확인합니다. x/y 뒤집힘·다른 사람의 점·새로 지어낸 값은 거부합니다.

**경계:** 점수 0.5는 통과하고 0.49는 거부합니다. 관절 `xy=None`, NaN, Inf, bool, 영역 밖, 다른 frame은 거부합니다. 응답의 `center=[0.501,0.299]`는 fixture의 `[0.5,0.3]`에서 각 축 오차 0.001이므로 허용합니다. 일치 검증 이후 이 함수가 반환하는 `center`부터 원본 검출 좌표 `[0.5,0.3]`로 바꿉니다. 원본을 사용하는 정책을 ROS에만 맡겨 두지 않습니다.

**완료 기준:** 정상 FOLLOW와 STOP을 모두 통과시키고, 조작한 frame·관절·좌표·중복 key를 아래 검증에서 거부해야 합니다. 항상 `None`을 반환하면 올바른 구현이 아닙니다.

**설정값과 부동소수점 경계:** `min_keypoint_conf`는 bool이 아닌 0–1 유한수, `tolerance`는 bool이 아닌 0–0.01 유한수이고 `expected_keypoint`는 위 17개 이름 중 하나입니다. 잘못된 설정이면 `None`입니다. 축별 경계 비교에는 표현 오차만 흡수하는 `1e-12`를 허용해 `[0.502,0.3]`와 기본 코의 차이가 0.002인 조건도 통과시킵니다. 이 작은 오차를 제어 좌표 허용 범위를 늘리는 값으로 사용하지 않습니다.


In [ ]:
student_pose_llm = load_file("student_pose_llm", "exercises/pose_llm_contract.py")
def check_pose_decision(validate):
    before = deepcopy(POSE_OBSERVATION)
    parsed = validate(POSE_RAW, POSE_OBSERVATION)
    assert parsed == POSE_DECISION, "동일 frame의 유효한 코 FOLLOW를 허용해야 합니다."
    assert POSE_OBSERVATION == before
    assert validate(json.dumps(POSE_STOP), POSE_OBSERVATION) == POSE_STOP
    assert validate(POSE_RAW, {**POSE_OBSERVATION,"person":None}) is None
    assert validate(json.dumps(POSE_STOP), {**POSE_OBSERVATION,"person":None}) == POSE_STOP
    cases = [
        {**POSE_DECISION,"frame_id":"pose-f0"},
        {**POSE_DECISION,"keypoint":"left_wrist"},
        {**POSE_DECISION,"center":[0.6,0.3]},
        {**POSE_DECISION,"center":[True,0.3]},
        {**POSE_DECISION,"center":[float('nan'),0.3]},
        {**POSE_DECISION,"center":[1.1,0.3]},
        {**POSE_DECISION,"command":"RUN"},
        {**POSE_DECISION,"speed":1.0},
        {**POSE_DECISION,"reason":""},
        {**POSE_STOP,"center":[0.5,0.3]},
        {**POSE_STOP,"keypoint":"nose"},
    ]
    for wrong in cases:
        assert validate(json.dumps(wrong), POSE_OBSERVATION) is None, wrong
    assert validate(POSE_RAW[:-1]+', "command":"STOP"}', POSE_OBSERVATION) is None, "중복 key"
    assert validate("```json\n"+POSE_RAW+"\n```", POSE_OBSERVATION) is None
    assert validate("not JSON", POSE_OBSERVATION) is None
    assert validate("[]", POSE_OBSERVATION) is None
    for confidence, xy in [(0.49,[0.5,0.3]),(0.9,None),(True,[0.5,0.3]),(0.9,[float('inf'),0.3])]:
        obs = deepcopy(POSE_OBSERVATION)
        obs["person"]["keypoints"]["nose"] = {"confidence":confidence,"xy":xy}
        assert validate(POSE_RAW, obs) is None
    threshold = deepcopy(POSE_OBSERVATION)
    threshold["person"]["keypoints"]["nose"]["confidence"] = 0.5
    assert validate(POSE_RAW, threshold) == POSE_DECISION
    near = {**POSE_DECISION,"center":[0.501,0.299]}
    assert validate(json.dumps(near), POSE_OBSERVATION) == POSE_DECISION, "반올림 응답을 승인한 뒤 반환 center는 검출 원본으로 되돌립니다."
    boundary = {**POSE_DECISION,"center":[0.502,0.3]}
    assert validate(json.dumps(boundary), POSE_OBSERVATION) == POSE_DECISION
    too_far = {**POSE_DECISION,"center":[0.5021,0.3]}
    assert validate(json.dumps(too_far), POSE_OBSERVATION) is None
    left = {**POSE_DECISION,"keypoint":"left_wrist","center":[0.5,0.5]}
    assert validate(json.dumps(left), POSE_OBSERVATION, expected_keypoint="left_wrist") == left
    for settings in ({"expected_keypoint":"finger_tip"},{"tolerance":0.1},{"tolerance":True},{"min_keypoint_conf":float("nan")}):
        assert validate(POSE_RAW, POSE_OBSERVATION, **settings) is None
    # 출력과 관측의 내부 list를 바꾸지 않는지도 함께 검사합니다.
    assert POSE_OBSERVATION == before
    print("PASS: JSON·동일 프레임·지시 관절·좌표 근거·점수·STOP·변조 거부")
check_pose_decision(student_pose_llm.validate_pose_decision)


## 참고 정답 검증

독립적인 [solutions/pose_llm_contract.py](solutions/pose_llm_contract.py)를 같은 검증에 적용합니다. 정답을 읽고 학생 파일에 반영한 뒤 위 학생 검증을 다시 실행합니다.


In [ ]:
reference_pose_llm = load_file("reference_pose_llm", "solutions/pose_llm_contract.py")
check_pose_decision(reference_pose_llm.validate_pose_decision)


### 정답 해설 — 문법 통과와 행동 근거를 따로 확인

**정답 파일:** [solutions/pose_llm_contract.py](solutions/pose_llm_contract.py)

첫 검사는 문법입니다. `json.loads`만으로는 중복 key를 발견하지 못하고 뒤 값을 덮어쓰므로, key/value 쌍을 받는 helper에서 중복을 검사합니다. JSON 바깥 문장과 코드 펜스도 그대로 거부합니다. “FOLLOW라고 쓰여 있다”는 문자열 검색으로 파싱하지 않습니다. 다섯 필드만 허용해 좌표와 관계없는 speed 같은 모델 지시가 실행 경로로 새어 들어가지 않게 합니다.

두 번째는 의미입니다. 정상 STOP은 `keypoint`와 `center`를 비웁니다. FOLLOW에는 요청한 `nose`가 반드시 있어야 하며 그 관절을 **이번 요청에 보낸 person 내부** 에서 찾습니다. frame_id를 먼저 대조해야 과거 사람의 코를 현재 프레임에 억지로 적용하지 않습니다. 어떤 값을 반환해야 할지 불명확하면 `None`으로 검증 실패를 알립니다.

세 번째는 측정 근거입니다. person 검출 점수가 높아도 코의 점수는 낮을 수 있습니다. 해당 코의 confidence가 0.49이면 얼굴이 있다고 가정하지 않고 FOLLOW를 거부합니다. `xy=None`은 미검출이므로 `(0,0)`으로 변환하지 않습니다. bool은 Python에서 숫자의 하위 타입이므로 별도로 거부하고 NaN·Inf도 유한수 검사로 거부합니다.

네 번째는 좌표 일치입니다. 모델이 반환한 `[0.5,0.3]`이 실제 코 `[0.5,0.3]`과 같은지 축별로 비교합니다. `[0.501,0.299]`의 작은 반올림은 허용하지만 `[0.6,0.3]`은 x 오차가 0.1이어서 거부합니다. 오차 허용은 프레임이나 관절 선택을 느슨하게 만드는 옵션이 아닙니다. 검증을 통과하면 이 함수의 반환값부터 원본 검출 좌표로 바꿉니다. 예를 들어 `[0.501,0.299]`를 읽고도 반환 `center`는 `[0.5,0.3]`이며 ROS도 그 검출 좌표를 사용합니다.

마지막으로 정상 JSON 승인과 실제 이동은 다릅니다. 승인 TTL 만료, 오래된 frame, 거북이 pose 중단, 목표 도착이면 행동 제어기는 STOP을 냅니다. 따라서 `decision=FOLLOW`와 overlay의 빨간 점이 동시에 나타날 수 있습니다. 모델 응답·최종 행동·정지 이유를 함께 읽어야 원인을 구분할 수 있습니다.

**이것이 LLM을 넣는 이유:** 고정 코 추종은 `person['keypoints']['nose']`를 직접 읽는 코드로도 가능합니다. 이 실습은 자연어 지시를 구조화된 대상 선택으로 바꾸고 검출 좌표로 검증하는 과정입니다. 마지막 관절 변경 실험에서 모델의 선택과 설정된 허용 대상을 함께 바꿔 보며 역할을 확인합니다. 좌표 JSON이라고 자동으로 더 빠르다고 가정하지 않고 실제 기록으로 비교합니다.

<!-- pose-decision-answer -->
<details>
<summary>참고 정답 전체 코드 — 단계별 주석 포함</summary>

[실제 참고 파일](solutions/pose_llm_contract.py)의 독립 구현입니다. 이 Markdown 코드는 자동 실행되지 않습니다. 학생 파일의 함수 시그니처를 유지하고 필요한 보조 함수도 함께 작성하세요.

```python
"""참고 정답: LLM 응답의 형식과 관측 증거를 단계별로 검사합니다.

1. json.loads(object_pairs_hook=...)에서 중복 키를 거부합니다. dict로 읽은 뒤에는
   앞의 STOP이 뒤의 FOLLOW로 덮였는지 알 수 없습니다. NaN/Inf도 JSON 로딩 때 거부합니다.
2. 다섯 키만 허용하고 각 필드를 검사합니다. STOP이면 keypoint와 center는 모두 None,
   FOLLOW이면 허용된 관절 이름과 0~1 좌표 두 개가 있어야 합니다. bool은 좌표가 아닙니다.
3. STOP도 frame_id를 대조합니다. 다른 프레임의 명령을 현재 관측으로 설명하지 않습니다.
4. FOLLOW는 요청 관절(expected_keypoint)과 관측의 같은 관절을 대조합니다.
   예를 들어 코를 요청했는데 손목 좌표를 보낸 응답은 유효한 숫자여도 거부합니다.
5. 관절 신뢰도는 임계값 이상이어야 합니다. 없는 관절/낮은 신뢰도 좌표를 추측하지 않습니다.
6. 축별 허용 오차(tolerance, 기본0.002)는 소수점 반올림을 위한 값입니다. 비교가
   통과해도 실제 반환에는 LLM 숫자 대신 관측 원본 xy를 사용합니다. 1e-12는 부동소수점
   경계 비교의 표현 오차만 흡수합니다. 임의의 이동 좌표를 승인하는 범위가 아닙니다.
7. 이 함수는 촬영 시각이나 승인 만료를 판단하지 않습니다. 반환된 결정도 매 제어 tick에
   max_age/TTL/pose_age를 검사해야 합니다. 프레임 일치와 시간 유효성은 별개의 검사입니다.

실제 ROS 모듈을 호출하지 않아 이 파일만으로 각 검사를 확인할 수 있습니다.
"""
import json
import math

def number(value):
    # Python의 bool은 int 하위형이므로 먼저 거부해야 True=1로 처리되지 않습니다.
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
        raise ValueError('finite number required')
    return float(value)


COCO_KEYPOINTS = (
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle',
)

POSE_SCHEMA = {
    'type': 'object',
    'properties': {
        'command': {'type': 'string', 'enum': ['FOLLOW', 'STOP']},
        'keypoint': {'anyOf': [{'type': 'string', 'enum': list(COCO_KEYPOINTS)}, {'type': 'null'}]},
        'center': {'anyOf': [{'type': 'array', 'items': {'type': 'number', 'minimum': 0, 'maximum': 1},
                             'minItems': 2, 'maxItems': 2}, {'type': 'null'}]},
        'frame_id': {'type': 'string', 'minLength': 1},
        'reason': {'type': 'string', 'minLength': 1, 'maxLength': 160},
    },
    'required': ['command', 'keypoint', 'center', 'frame_id', 'reason'],
    'additionalProperties': False,
}

def unit_number(value):
    value = number(value)
    if not 0 <= value <= 1:
        raise ValueError('normalized value outside [0,1]')
    return value

def unit_xy(value):
    if not isinstance(value, (list, tuple)) or len(value) != 2:
        raise ValueError('xy must contain two normalized coordinates')
    return [unit_number(v) for v in value]

def parse_pose_decision(raw):
    """Reject ambiguous JSON and contradictory STOP/FOLLOW fields."""
    def unique_object(pairs):
        value = {}
        for key, item in pairs:
            if key in value:
                raise ValueError('duplicate JSON key')
            value[key] = item
        return value

    def bad_constant(value):
        raise ValueError(f'nonfinite JSON number: {value}')

    try:
        value = json.loads(raw, object_pairs_hook=unique_object, parse_constant=bad_constant)
        if not isinstance(value, dict) or set(value) != set(POSE_SCHEMA['required']):
            return None
        if value['command'] not in ('FOLLOW', 'STOP'):
            return None
        if not isinstance(value['frame_id'], str) or not value['frame_id'].strip():
            return None
        if not isinstance(value['reason'], str) or not 1 <= len(value['reason'].strip()) <= 160:
            return None
        if value['command'] == 'STOP':
            return value if value['keypoint'] is None and value['center'] is None else None
        if value['keypoint'] not in COCO_KEYPOINTS:
            return None
        value['center'] = unit_xy(value['center'])
        return value
    except (ValueError, TypeError, KeyError, OverflowError):
        return None

def validate_pose_decision(raw, observation, *, expected_keypoint='nose',
                           min_keypoint_conf=0.5, tolerance=0.002):
    """Approve a coordinate only with matching frame, requested joint and evidence.

Axis-wise tolerance is inclusive and permits decimal rounding only. Return the
original detector coordinate, never the LLM's rounded or invented coordinate.
Time and turtlesim pose checks happen on EVERY controller tick in pose_motion.
"""
    try:
        threshold = unit_number(min_keypoint_conf)
        epsilon = number(tolerance)
        if expected_keypoint not in COCO_KEYPOINTS or not 0 <= epsilon <= 0.01:
            return None
        value = parse_pose_decision(raw)
        if value is None or value['frame_id'] != observation['frame_id']:
            return None
        if value['command'] == 'STOP':
            return value
        if value['keypoint'] != expected_keypoint:
            return None
        joint = observation['person']['keypoints'][expected_keypoint]
        xy = unit_xy(joint['xy'])
        if unit_number(joint['confidence']) < threshold:
            return None
        if any(abs(actual - claimed) > epsilon + 1e-12 for actual, claimed in zip(xy, value['center'])):
            return None
        return {**value, 'center': xy}
    except (ValueError, TypeError, KeyError, OverflowError):
        return None
```

</details>



## 실제 ROS 실행 — JupyterLab 터미널 A·B·C

<!-- procedure:ros-terminals-57bc75:start -->
![새 터미널 준비와 세 역할](../교재그림/절차/ros-terminals.svg)

1–3을 새 터미널마다 적용합니다. 4–6은 동시에 실행하는 역할이며 노드·토픽 이름은 해당 실습 명령을 따릅니다.
<!-- procedure:ros-terminals-57bc75:end -->


01–04와 같은 로컬 실습 컨테이너에서 실행합니다. 05의 `live_yolo.py` 웹캠 실행과 기존 손 추종 제어기는 먼저 종료합니다. **새 터미널은 첫 줄 `bash`부터 전체 블록** 을 복사합니다. A·B·C의 Domain은 배정값(예:21)으로 맞춥니다. 이미 설정한 같은 터미널에서 이어 실행할 때는 아래에 표시한 재실행 launch만 사용합니다.

**처음 준비 또는 package not found일 때 — 빌드:**
```bash
bash
bash /workspace/08.ROS/scripts/build_workspace.sh
bash /workspace/09.Pseudo-VLA/scripts/build_workspace.sh
```

`yolo26n-pose.pt`와 공용 Ollama의 `gemma4:e2b-it-qat`가 준비되어 있어야 합니다. 같은 모델을 사용하더라도 이번 요청에는 `images`가 없고 좌표 JSON 텍스트만 들어갑니다. 모델 요청 실패를 정답 응답으로 대신하지 않습니다.

**A — 카메라 입력:** 기존 A가 살아 있으면 추가 실행하지 않습니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 run v4l2_camera v4l2_camera_node --ros-args -p video_device:=/dev/video0 -p image_size:="[640, 480]" -r image_raw:=/camera/image_raw
```

**B — 가장 큰 사람의 코 추종:** 한 번의 launch로 pose·LLM 제어기, turtlesim, overlay 뷰어가 실행됩니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 launch nepes_vla_demo pose_follow.launch.py \
  pose_model:=/workspace/Data/models/yolo26n-pose.pt \
  llm_model:=gemma4:e2b-it-qat \
  target_keypoint:=nose \
  instruction:="Follow the nose of this person. Stop if the nose is invalid." \
  ttl_ms:=400 \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA/06-pose-nose.jsonl
```

**C — 최종 행동:** 새 터미널에서 전체 블록을 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 topic echo /pose/action
```

C의 echo를 Ctrl+C로 종료한 뒤 **같은 C에서** 입력과 판단을 하나씩 봅니다.
```bash
ros2 topic echo /pose/perception
```
위 echo를 Ctrl+C로 종료한 **같은 C에서**:
```bash
ros2 topic echo /pose/decision
```

**화면 읽기:** `/pose/image_overlay`에는 선택된 사람의 박스·관절과 선택한 추종 점이 표시됩니다. 초록은 전진·회전 중, 빨강은 유효한 좌표가 있지만 정지 중입니다. 유효 좌표가 없으면 `NO TARGET`입니다. frame·승인 경과시간·TTL·정지 이유를 함께 읽습니다. 추론에 사용한 영상과 점을 짝지으므로 LLM을 기다리는 동안 화면도 지난 관측을 유지할 수 있습니다. 카메라 원본의 최신 화면과 판단에 사용한 영상은 다를 수 있습니다.

**움직임:** 코를 영상 오른쪽/위로 이동하면 거북이 목표도 오른쪽/위로 이동합니다. 목표와 방향 차이가 크면 먼저 회전하고, 목표에 도착하면 멈춥니다. 가장 큰 사람이 바뀌면 목표 인물도 바뀝니다. 그 사람의 코가 가려져 유효하지 않으면 STOP입니다. 이번 경로는 손가락을 보지 않으므로 주먹을 쥐는 동작은 정지 명령이 아닙니다.

**종료:** A를 멈춘 뒤 C에서 시간 제한에 의한 STOP을 확인하고 B·C를 종료합니다. 다음 조건을 비교할 때는 A를 유지하고 B만 Ctrl+C로 종료합니다. GUI 없는 자습은 B의 launch 앞에 `QT_QPA_PLATFORM=offscreen`, 인자에 `with_image_view:=false`를 사용합니다.

학생 TODO는 제공 검증 셀에서 평가하고, 실제 ROS는 제공된 실행 구현을 사용합니다. TODO를 저장해도 실행 중인 제어기의 구현이 자동으로 바뀌지는 않습니다.


## 관찰 실험 — 같은 사람·다른 지시·불확실한 입력

각 조건에서 **원본 자세, 선택한 사람, 입력 관절 confidence, LLM 응답, 최종 행동·reason, latency_ms** 를 함께 기록합니다.

1. 한 사람의 코를 좌·중앙·우로 이동합니다. 목표 `(1+9*x,1+9*(1-y))`와 점 색을 확인합니다.
2. 두 사람이 가까이/멀리 이동합니다. bbox 면적 역전에서 선택된 사람이 바뀌는지 확인합니다. 이 기능은 특정 사람의 identity를 유지하는 tracker가 아닙니다.
3. 선택된 사람의 코를 가리거나 프레임 밖으로 옮깁니다. 모델이 FOLLOW를 말해도 좌표 근거·신뢰도가 없으면 검증으로 STOP이 되는지 봅니다. 검출 모델이 가려진 관절을 높은 점수로 잘못 추정하는 경우도 기록하며, 가림이 항상 낮은 점수를 보장한다고 가정하지 않습니다.
4. B를 Ctrl+C로 종료하고 **같은 B에서** 다음을 실행합니다. 기본 코 대신 `left_wrist`를 지시하고 검증에 허용할 대상도 함께 바꿉니다. ROS 프로그램을 고칠 필요는 없습니다.

```bash
ros2 launch nepes_vla_demo pose_follow.launch.py \
  pose_model:=/workspace/Data/models/yolo26n-pose.pt \
  llm_model:=gemma4:e2b-it-qat \
  target_keypoint:=left_wrist \
  instruction:="Follow the left wrist of this person. Stop if the left wrist is invalid." \
  ttl_ms:=400 \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA/06-pose-left-wrist.jsonl
```

`target_keypoint`는 검증용 허용 대상입니다. 지시만 손목으로 바꾸고 허용 대상을 nose로 둔 채 모델이 손목을 반환하면 의도적으로 거부됩니다. 고정 nose 코드로도 해결되는 기본 과제와, 자연어 대상 선택·검증이라는 LLM의 역할을 구분하세요.

**비교 조건:** 03의 이미지 VLM 손 추종과 이번 좌표 LLM은 인지 모델·대상·입력 표현이 다릅니다. 같은 400ms TTL이어도 판단 속도가 같다는 보장은 없습니다. 기록한 요청 지연과 STOP 이유를 비교하되, 이 실험만으로 LLM과 VLM의 일반 성능 우열을 주장하지 않습니다.


## 학생 검증 결과 저장

아래는 실제 모델 측정이 아닌 **학생 함수 fixture 검증 결과** 입니다. 항상 재검증한 뒤 저장합니다. 실제 ROS 관찰은 별도 표에 기록하고, 수행하지 않은 항목은 미측정으로 남깁니다.


In [ ]:
check_pose_decision(student_pose_llm.validate_pose_decision)
fixture_result = {
    "kind":"student-pose-decision-fixture", "measured_model_latency":False,
    "observation":POSE_OBSERVATION,
    "validated":student_pose_llm.validate_pose_decision(POSE_RAW, POSE_OBSERVATION),
    "turtle_target":[5.5,7.3],
}
fixture_path = ROOT / "Data/outputs/09.Pseudo-VLA/pose-llm-fixture.json"
fixture_path.parent.mkdir(parents=True, exist_ok=True)
fixture_path.write_text(json.dumps(fixture_result, ensure_ascii=False, indent=2)+"\n")
print("저장:", fixture_path)


## 실험 기록 TODO — 실제 관측을 적기

아래 빈칸은 관찰·보고 TODO입니다. 위 `validate_pose_decision` 함수 구현과 구분합니다. 최소 한 조건의 기록에서 입력과 최종 행동을 대조하고, 아직 실행하지 못한 조건은 `미측정`으로 둡니다. latency 숫자를 fixture 값으로 채우지 않습니다.

| 조건 | 선택 박스·관절 | 모델 응답 | 최종 행동·reason | 관측/판단 지연 | 해석 |
|---|---|---|---|---|---|
| 코 왼쪽/오른쪽 이동 | 미측정 | 미측정 | 미측정 | 미측정 | 미측정 |
| 두 사람 면적 순위 전환 | 미측정 | 미측정 | 미측정 | 미측정 | 미측정 |
| 코 가림/미검출 | 미측정 | 미측정 | 미측정 | 미측정 | 미측정 |
| 왼쪽 손목 지시로 변경 | 미측정 | 미측정 | 미측정 | 미측정 | 미측정 |

**예상 해석:** 유효한 코 FOLLOW는 코를 목표로 승인할 수 있습니다. 승인 만료·목표 도착이면 같은 FOLLOW라도 최종 속도는 0입니다. 면적 순위가 바뀌면 새 사람을 선택하며, 코가 유효하지 않으면 작은 사람으로 대신 이동하지 않습니다. 지시와 허용 대상이 일치한 손목 응답만 손목 추종으로 이어집니다. 실제 결과가 이 예상과 다르면 입력 JSON부터 원인을 찾습니다.


## 완료 확인과 공통 정리 / 별도 30분

- 내 `validate_pose_decision()`이 정상 FOLLOW/STOP과 잘못된 응답을 구분합니다.
- 최대 면적 사람의 17개 관절을 통째로 보내며 이미지가 요청에 포함되지 않음을 설명합니다.
- 모델 응답 좌표를 그대로 속도로 쓰지 않고 같은 frame의 검출 좌표와 대조하는 이유를 설명합니다.
- 코 추종·대상 전환·누락·손목 지시 실험의 실제 관찰과 미측정을 구분합니다.
- 학생 결과 `pose-llm-fixture.json`과 실제 ROS `06-pose-*.jsonl` 로그를 구분해 제출합니다.

[평가 워크북](../평가워크북.md)에서 01 규칙, 03 이미지 VLM, 04 TTL, 06 좌표 LLM의 입력·판단·최종 행동을 비교합니다. 5일차에는 이 수동 모듈 연결과, 영상·언어·상태에서 행동을 학습한 정책의 차이로 이어갑니다. [07 segmentation](07_yolo_segmentation.ipynb)은 선택 자습입니다.


<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

[4일차 슬라이드 64쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_pose_llm_01) · [배포 PDF 64쪽](../슬라이드/4일차.pdf#page=64)에서 64–71쪽을 복습합니다. 자신의 TODO 검증과 실제 관측·미측정을 나누어 설명합니다.
<!-- slide-sync:end -->
